# Rankwise: Two-Stage Recommender with Counterfactual Evaluation
### End-to-End GPU Benchmark on KuaiRand-Pure Unbiased Traffic

This notebook reproduces the complete **Rankwise** pipeline on Google Colab with GPU acceleration:
1. **Dataset Acquisition**: KuaiRand-Pure Zenodo archive extraction & temporal splitting.
2. **Stage 1 (Retrieval)**: Two-Tower deep neural retriever with in-batch softmax and $\log Q$ popularity debiasing + FAISS HNSW index.
3. **Stage 2 (Ranking)**: Multi-Task DCN-v2 (Deep & Cross Network v2 for click & long-view) vs. LightGBM LambdaMART baseline.
4. **Calibration**: Isotonic probability calibration & Expected Calibration Error (ECE).
5. **Counterfactual OPE**: Propensity estimation, Clipped IPS, and Self-Normalized IPS (SNIPS) evaluated against held-out unbiased random traffic ground truth.
6. **Latency Profiling**: FastAPI serving inference latency (p50, p95, p99) and RPS throughput.

In [ ]:
# 1. Environment & GPU Check
import torch, platform
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Compute Device:', device)
if torch.cuda.is_available():
    print('GPU Model:', torch.cuda.get_device_name(0))
    print('VRAM:', f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

In [ ]:
# 2. Clone Repository & Install Dependencies
!git clone https://github.com/bhuvneshsahu01/rankwise.git || cd rankwise && git pull
%cd rankwise
!pip install -q torch lightgbm faiss-cpu onnx onnxscript onnxruntime fastapi uvicorn scikit-learn pandas numpy matplotlib seaborn

In [ ]:
# 3. Download & Extract KuaiRand-Pure Dataset
import os, tarfile, urllib.request, hashlib

data_raw_dir = 'data_raw'
os.makedirs(data_raw_dir, exist_ok=True)
tar_path = os.path.join(data_raw_dir, 'KuaiRand-Pure.tar.gz')

if not os.path.exists(tar_path):
    print('[*] Downloading KuaiRand-Pure dataset (47.4 MB)...')
    zenodo_url = 'https://zenodo.org/records/10439422/files/KuaiRand-Pure.tar.gz'
    !curl -L -o {tar_path} {zenodo_url}

extract_dir = os.path.join(data_raw_dir, 'extracted')
if not os.path.exists(extract_dir):
    print('[*] Extracting dataset...')
    with tarfile.open(tar_path, 'r:gz') as tar:
        tar.extractall(extract_dir)
print('[✓] Dataset ready!')

In [ ]:
# 4. Execute Full Reproducible Benchmark
!python benchmarks/run_rankwise_benchmarks.py

In [ ]:
# 5. Inspect Results JSON
import json
with open('benchmarks/results/rankwise_metrics.json', 'r') as f:
    results = json.load(f)
print(json.dumps(results, indent=2))